# z分数、百分位数与箱形图完整教程：从相对位置到数据可视化

## 📚 教学目标
1. 理解 z 分数的定义：$z = \frac{x - \bar{x}}{s}$，衡量数据点的相对位置
2. 掌握百分位数和四分位数的计算方法
3. 学会构建五数概括法和箱形图
4. 用箱形图识别异常值（IQR 方法）
5. 比较不同数据集的分布特征

**参考书**：《基础统计学(第14版)》（Triola）第 3-3 节
**教学策略**：先用极小数据集让你「看见」每一步计算，再扩展到真实规模

---

## 1. 为什么需要相对位置的度量？

### 🎯 一个直觉问题

小明考了 90 分，小红考了 70 分。谁的成绩更好？

答案是：**取决于整体分布！**
- 如果全班均值 85 分、标准差 5 分 → 小明的 90 分只比均值高 1 个标准差
- 如果全班均值 60 分、标准差 10 分 → 小红的 70 分比均值高 1 个标准差

小红的成绩在其班级中**相对位置更高**！

### 📖 书中的定义

> z 分数（或标准分数）是指给定的数据值 x 高于或低于均值的标准差的个数。
> z 分数是一种纯数字标记，即没有测量单位。

接下来我们将学习：**z 分数** → **百分位数** → **四分位数** → **箱形图**。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print('✅ 库导入完成')

---

## 2. z 分数：标准化的相对位置

### 📐 计算公式

$$z = \frac{x - \bar{x}}{s}$$

其中：
- $x$ = 数据值
- $\bar{x}$ = 样本均值
- $s$ = 样本标准差

### 💡 z 分数的含义

| z 分数 | 含义 |
|--------|------|
| $z > 0$ | 数据值高于均值 |
| $z < 0$ | 数据值低于均值 |
| $z = 0$ | 数据值等于均值 |
| $z \geq 2$ 或 $z \leq -2$ | 显著高或显著低（经验法则） |

In [ ]:
wait_small = np.array([50, 25, 75, 35, 50, 25, 30, 50, 45, 25, 20])

wait_full = np.array([
    10, 15, 15, 15, 15, 15, 20, 20, 20, 20,
    25, 25, 25, 25, 25, 25, 30, 30, 30, 30,
    30, 30, 30, 30, 35, 35, 35, 35, 35, 35,
    35, 35, 40, 40, 40, 40, 45, 50, 50, 50,
    50, 50, 55, 55, 60, 75, 75, 75, 105, 110
])

print('📋 微型数据集（前 11 个）：')
print(f'  {wait_small}')
print(f'  均值 x̄ = {wait_small.mean():.2f}, 标准差 s = {wait_small.std(ddof=1):.2f}')
print(f'\n📋 完整数据集（50 个）：')
print(f'  均值 x̄ = {wait_full.mean():.2f}, 标准差 s = {wait_full.std(ddof=1):.2f}')

In [ ]:
x_bar = wait_small.mean()
s = wait_small.std(ddof=1)

print('📊 手算 z 分数（微型数据集）')
print('=' * 55)
print(f'  均值 x̄ = {x_bar:.4f}')
print(f'  标准差 s = {s:.4f}')
print()

print(f'  {"x_i":>6} | {"x - x̄":>10} | {"z = (x-x̄)/s":>14} | {"解读"}')
print(f'  {"-"*50}')

for x in sorted(wait_small):
    z = (x - x_bar) / s
    if z >= 2:
        interpretation = '⚠️ 显著高'
    elif z <= -2:
        interpretation = '⚠️ 显著低'
    else:
        interpretation = '正常'
    print(f'  {x:>6} | {x - x_bar:>10.4f} | {z:>14.4f} | {interpretation}')

In [ ]:
z_scores = stats.zscore(wait_small, ddof=1)

print('🔬 scipy.stats.zscore 验证:')
print(f'  {"x_i":>6} | {"手算 z":>10} | {"scipy z":>10} | {"一致？"}')
print(f'  {"-"*45}')
for x, z_manual, z_scipy in zip(wait_small, [(x - x_bar)/s for x in wait_small], z_scores):
    match = '✅' if abs(z_manual - z_scipy) < 1e-10 else '❌'
    print(f'  {x:>6} | {z_manual:>10.4f} | {z_scipy:>10.4f} | {match}')
print(f'\n  ✅ 验证通过！')

---

## 3. 百分位数：位置的另一种度量

### 📐 定义

百分位数将数据分为 100 组，每组约 1% 的数据值。

### 📐 求数据值的百分位数

$$\text{百分位数} = \frac{\text{小于 } x \text{ 的数据值个数}}{n} \times 100$$

### 📐 将百分位数转换为数据值

1. 计算位置：$L = \frac{k}{100} \times n$
2. 如果 $L$ 是整数，取第 $L$ 个和第 $L+1$ 个值的平均
3. 如果 $L$ 不是整数，向上取整到下一个整数位置

In [ ]:
sorted_data = np.sort(wait_full)
n_full = len(sorted_data)

print('📊 步骤 1: 求等候时间 45 分钟的百分位数')
print('-' * 50)
x_target = 45
count_below = np.sum(sorted_data < x_target)
percentile = (count_below / n_full) * 100
print(f'  小于 {x_target} 的数据值个数 = {count_below}')
print(f'  百分位数 = ({count_below} / {n_full}) x 100 = {percentile:.0f}')
print(f'  💡 {x_target} 分钟位于第 {percentile:.0f} 百分位')

In [ ]:
def percentile_to_value(data, k):
    n = len(data)
    L = (k / 100) * n
    if L == int(L):
        L = int(L)
        value = (data[L-1] + data[L]) / 2
    else:
        L = int(np.ceil(L))
        value = data[L-1]
    return L, value

print('📊 步骤 2: 将百分位数转换为数据值')
print('-' * 50)

for k in [25, 50, 90]:
    L, value = percentile_to_value(sorted_data, k)
    print(f'  P_{k}: L = ({k}/100) x {n_full} = {(k/100)*n_full}')
    if (k/100)*n_full == int((k/100)*n_full):
        print(f'         L 是整数，取第 {L} 和第 {L+1} 个值的平均')
    else:
        print(f'         L 不是整数，向上取整得 {L}')
    print(f'         P_{k} = {value} 分钟')
    print()

In [ ]:
print('🔬 scipy 验证百分位数:')
for k in [25, 50, 90]:
    manual_val = percentile_to_value(sorted_data, k)[1]
    scipy_val = np.percentile(wait_full, k)
    print(f'  P_{k}: 手算 = {manual_val}, scipy = {scipy_val}')
print(f'\n  💡 scipy 默认使用线性插值法，可能与手算结果略有不同')

---

## 4. 四分位数与五数概括法

### 📐 四分位数定义

| 四分位数 | 等于 | 含义 |
|---------|------|------|
| $Q_1$ | $P_{25}$ | 25% 数据低于此值 |
| $Q_2$ | $P_{50}$ | 中位数 |
| $Q_3$ | $P_{75}$ | 75% 数据低于此值 |

### 📐 四分位距 (IQR)

$$\text{IQR} = Q_3 - Q_1$$

### 📐 异常值判断（IQR 方法）

- **异常值**：$x < Q_1 - 1.5 \times \text{IQR}$ 或 $x > Q_3 + 1.5 \times \text{IQR}$

In [ ]:
Q1 = np.percentile(wait_full, 25)
Q2 = np.percentile(wait_full, 50)
Q3 = np.percentile(wait_full, 75)
data_min = wait_full.min()
data_max = wait_full.max()
IQR = Q3 - Q1

print('📊 五数概括法（完整 50 个数据）')
print('=' * 45)
print(f'  最小值  = {data_min}')
print(f'  Q1 (P25) = {Q1}')
print(f'  Q2 (中位数) = {Q2}')
print(f'  Q3 (P75) = {Q3}')
print(f'  最大值  = {data_max}')
print(f'\n📐 四分位距 (IQR)')
print(f'  IQR = Q3 - Q1 = {Q3} - {Q1} = {IQR}')
print(f'\n📐 异常值边界')
print(f'  下界 = Q1 - 1.5 x IQR = {Q1} - 1.5 x {IQR} = {Q1 - 1.5*IQR}')
print(f'  上界 = Q3 + 1.5 x IQR = {Q3} + 1.5 x {IQR} = {Q3 + 1.5*IQR}')

In [ ]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = wait_full[(wait_full < lower_bound) | (wait_full > upper_bound)]
non_outliers = wait_full[(wait_full >= lower_bound) & (wait_full <= upper_bound)]

print('📊 异常值检测')
print('=' * 45)
print(f'  异常值边界: [{lower_bound:.1f}, {upper_bound:.1f}]')
print(f'  异常值: {outliers}')
print(f'  非异常值的最小值: {non_outliers.min()}')
print(f'  非异常值的最大值: {non_outliers.max()}')
print(f'\n💡 原书结果：异常值为 105 和 110 分钟')

In [ ]:
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
bp1 = ax1.boxplot(wait_full, vert=False, widths=0.6, patch_artist=True,
                  boxprops=dict(facecolor='steelblue', alpha=0.7),
                  medianprops=dict(color='#e74c3c', linewidth=2))
ax1.set_xlabel('Wait Time (min)', fontsize=12)
ax1.set_title('Box Plot: Space Mountain Wait Times', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

ax2 = axes[1]
bp2 = ax2.boxplot(wait_full, vert=False, widths=0.6, patch_artist=True,
                  boxprops=dict(facecolor='steelblue', alpha=0.7),
                  medianprops=dict(color='#e74c3c', linewidth=2),
                  flierprops=dict(marker='X', markerfacecolor='#e74c3c', markersize=10, markeredgecolor='black'))
ax2.set_xlabel('Wait Time (min)', fontsize=12)
ax2.set_title('Modified Box Plot with Outliers', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

for o in outliers:
    ax2.annotate(f'Outlier={o}', xy=(o, 1), xytext=(o, 0.65),
                fontsize=9, ha='center', color='#e74c3c', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/3.3_fig1.png', dpi=100, bbox_inches='tight')
plt.close()
print('✅ 图表已保存')

print(f'\n💡 左图：常规箱形图，展示五数概括法')
print(f'   右图：修正箱形图，红色 X 标记异常值（105 和 110 分钟）')
print(f'   箱子从 Q1={Q1} 到 Q3={Q3}，中位数为 {Q2}')

---

## 5. 扩展到 500 个数据点

In [ ]:
n_large = 500

# 模拟正态分布的考试成绩：均值 75，标准差 12
scores = np.random.normal(loc=75, scale=12, size=n_large)
scores = np.clip(scores, 0, 100)

mean_scores = scores.mean()
std_scores = scores.std(ddof=1)

q1 = np.percentile(scores, 25)
q2 = np.percentile(scores, 50)
q3 = np.percentile(scores, 75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers_large = scores[(scores < lower) | (scores > upper)]

print('=' * 55)
print('📋 500 个考试成绩的描述性统计')
print('=' * 55)
print(f'\n📊 基本统计量:')
print(f'  样本量 n     = {n_large}')
print(f'  均值 x̄       = {mean_scores:.2f}')
print(f'  标准差 s     = {std_scores:.2f}')
print(f'\n📐 五数概括法:')
print(f'  最小值       = {scores.min():.2f}')
print(f'  Q1          = {q1:.2f}')
print(f'  Q2 (中位数)  = {q2:.2f}')
print(f'  Q3          = {q3:.2f}')
print(f'  最大值       = {scores.max():.2f}')
print(f'  IQR         = {iqr:.2f}')
print(f'\n📐 异常值检测:')
print(f'  边界: [{lower:.2f}, {upper:.2f}]')
print(f'  异常值个数: {len(outliers_large)}')

---

## 6. 比较不同数据集的箱形图

In [ ]:

space_mountain = np.random.normal(40, 20, 50).clip(5, 150)
terror = np.random.normal(40, 10, 50).clip(10, 80)
avatar = np.random.normal(80, 30, 50).clip(20, 250)

datasets = [space_mountain, terror, avatar]
names = ['Space Mountain', 'Terror Tower', 'Avatar Flight']

print('📊 三个游乐项目等候时间的五数概括法')
print('=' * 65)
print(f'  {"项目":<20} {"Min":>6} {"Q1":>6} {"Q2":>6} {"Q3":>6} {"Max":>6} {"IQR":>6}')
print(f'  {"-"*56}')
for data, name in zip(datasets, names):
    q1, q2, q3 = np.percentile(data, [25, 50, 75])
    print(f'  {name:<20} {data.min():>6.1f} {q1:>6.1f} {q2:>6.1f} {q3:>6.1f} {data.max():>6.1f} {q3-q1:>6.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#3498db', '#2ecc71', '#e74c3c']
bp = ax.boxplot(datasets, labels=names, patch_artist=True, widths=0.5,
                flierprops=dict(marker='X', markerfacecolor='#e67e22', markersize=8, markeredgecolor='black'))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Wait Time (min)', fontsize=12)
ax.set_title('Disneyland Wait Times Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/3.3_fig2.png', dpi=100, bbox_inches='tight')
plt.close()
print('✅ 图表已保存')

print(f'\n💡 阿凡达飞行历险的等候时间最长（中位数远高于其他两个）')
print(f'   恐怖魔塔的离散程度最小（箱子最窄，IQR 最小）')

---

## 📌 核心概念回顾

### 📌 z 分数 (z-score)
- **公式**: $z = \frac{x - \bar{x}}{s}$
- **含义**: 标准化的位置度量，无量纲
- **判断标准**: $|z| \geq 2$ 为显著值

### 📌 百分位数 (Percentile)
- **公式**: $P_k$ 的位置 = $\frac{k(n+1)}{100}$
- **含义**: 第 $k$ 百分位数表示约 $k\%$ 的数据低于此值

### 📌 四分位数 (Quartile)
- $Q_1 = P_{25}$, $Q_2 = P_{50}$, $Q_3 = P_{75}$
- **IQR**: $\text{IQR} = Q_3 - Q_1$

### 📌 箱形图 (Box Plot)
- **五数概括法**: 最小值、$Q_1$、$Q_2$、$Q_3$、最大值
- **异常值**: 低于 $Q_1 - 1.5 \times \text{IQR}$ 或高于 $Q_3 + 1.5 \times \text{IQR}$

### 🔗 完整流程
```
原始数据 → 排序 → 五数概括法 → 计算 IQR → 判断异常值 → 绘制箱形图
```

---

## ❌ 常见误区

### ❌ 误区 1: z 分数有单位
**✓ 正确理解**: z 分数是无量纲的标准化值。分子与分母单位相同，相除后单位抵消。

### ❌ 误区 2: 百分位数的计算方法是唯一的
**✓ 正确理解**: 百分位数和四分位数的计算方法没有统一标准，不同软件可能给出不同结果。

### ❌ 误区 3: 箱形图的须线延伸到最小值和最大值
**✓ 正确理解**: **常规**箱形图延伸到最小值和最大值；**修正**箱形图只延伸到非异常值的极值。

### ❌ 误区 4: z 分数大于 2 一定是异常值
**✓ 正确理解**: $|z| \geq 2$ 表示「显著」值，但异常值的正式判断标准是 IQR 方法。

### ❌ 误区 5: 中位数总是在箱子的正中间
**✓ 正确理解**: 只有当数据关于中位数对称时才在正中间。偏态数据中中位数会偏向一侧。